# Lenta price-tag detector — one-time training on a free GPU

Inference stays **local CPU/ONNX** (ТЗ-compliant: no cloud in the running
solution). This notebook only produces `models/detector.onnx`.

**Runtime → Change runtime type → GPU (T4)**, then Run all.
At the end download `detector.onnx` into the repo's `models/`.

In [ ]:
# 1. Code + deps
!git clone https://github.com/REPLACE_WITH_YOUR_REPO lentaworld || true
%cd lentaworld
!pip -q install -r requirements.txt -r requirements-autolabel.txt gdown

In [ ]:
# 2. Data (the labeled folders + Unlabeled videos)
!gdown --folder https://drive.google.com/drive/folders/1XRrRB7y66RU4lxZiH7a6H_b8fOvKgOQl -O data

In [ ]:
# 3. Hybrid dataset: provided GT boxes + Grounding-DINO auto-labels
!python scripts/build_gt_dataset.py --data data --out dataset --val-zones 49_5
!python scripts/autolabel.py --data data --out dataset --stride 15 --score 0.35

In [ ]:
# 4. Train + export ONNX (FasterRCNN-R50, BSD-3; ~20-40 min on T4)
!python scripts/train_detector.py --dataset dataset --epochs 30 --out models/detector.onnx

In [ ]:
# 5. Download the weights -> put into local repo models/
from google.colab import files
files.download('models/detector.onnx')